In [15]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO, HIGIENIZAÇÃO E SALVAMENTO DA BASE LIMPA (NHIS 2024)
# ==============================================================================
import numpy as np
import pandas as pd

# 1. Dicionário Consolidado das Variáveis Selecionadas (87 Features + Controles)
MAPA_VARIAVEIS_PAPER = {
    "DEPEV_A": "historico_depressao",
    "DEPFREQ_A": "freq_depressao",
    "DEPLEVEL_A": "intensidade_depressao",
    "DEPMED_A": "med_depressao",
    "K6SPD_A": "sofrimento_psicologico_k6",
    "EFFORT_A": "k6_freq_esforco_extremo",
    "SAD_A": "k6_freq_tristeza_profunda",
    "SLPFLL_A": "dificuldade_adormecer",
    "HOPELESS_A": "k6_freq_desesperanca",
    "SLPSTY_A": "dificuldade_manter_sono",
    "SLPREST_A": "acorda_descansado",
    "FGEFRQTRD_A": "frequencia_fadiga_3m",
    "FDSLESS_A": "comeu_menos_que_devia",
    "FDSWEIGHT_A": "perdeu_peso_restricao_alimentar",
    "WORTHLESS_A": "k6_freq_desvalorizacao",
    "COGMEMDFF_A": "dificuldade_memoria_concentracao",
    "ANXEV_A": "historico_transtorno_ansiedade",
    "ANXFREQ_A": "freq_ansiedade",
    "ANXLEVEL_A": "intensidade_ansiedade",
    "ANXMED_A": "med_ansiedade",
    "NERVOUS_A": "k6_freq_nervosismo",
    "PAYWORRY_A": "estresse_financeiro_medico",
    "RESTLESS_A": "k6_freq_agitacao_inquieta",
    "LONELY_A": "frequencia_solidao",
    "SUPPORT_A": "suporte_social",
    "PCNTADLT_A": "qtd_adultos_familia",
    "SOCWRKLIM_A": "limitacao_trabalho_por_saude",
    "LSATIS4_A": "satisfacao_com_a_vida",
    "PARSTAT_A": "status_parental",
    "PERASST_A": "necessita_ajuda_outra_pessoa",
    "COGTYPEDFF_A": "tipo_dificuldade_cognitiva",
    "COGFRQDFF_A": "frequencia_falha_memoria",
    "NOEQWLK13M_A": "dificuldade_500m_sem_aparelho",
    "NOEQSTEPS_A": "dificuldade_degraus_sem_aparelho",
    "EQWLK13M_A": "dificuldade_500m_com_aparelho",
    "EQSTEPS_A": "dificuldade_degraus_com_aparelho",
    "REPSTRAIN_A": "lesao_por_esforco_repetitivo",
    "REPLIMIT_A": "limitacao_por_ler_dort",
    "ARTHEV_A": "historico_artrite",
    "REPWRKCAUS_A": "ler_dort_causada_no_trabalho",
    "TBIHLSBMC_A": "concussao_sintomas_pos_trauma",
    "INJFALL_A": "lesao_decorrente_queda",
    "INJFALLHOM_A": "queda_ocorrida_em_casa",
    "PAYNOBLLNW_A": "dividas_medicas_em_aberto",
    "RSNHICOST_A": "sem_plano_por_custo_inacessivel",
    "POVRATTC_A": "razao_renda_pobreza",
    "RATCAT_A": "categoria_razao_pobreza",
    "HOUSECOST_A": "dificuldade_custo_moradia",
    "HOUTENURE_A": "tipo_posse_imovel",
    "HOUYRSLIV_A": "tempo_moradia_anos",
    "FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    "FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    "FLUNCH12M1_A": "merenda_escolar_gratuita",
    "SEX_A": "sexo",
    "AGE65": "faixa_65_mais",
    "OVER65FLG_A": "flg_idoso_familia",
    "MARITAL_A": "estado_civil_declarado",
    "MARSTAT_A": "estado_civil",
    "SPOUSESEX_A": "sexo_conjuge",
    "SPOUSEP_A": "separacao_legal_conjugal",
    "REGION": "regiao_geografica",
    "URBRRL23": "classificacao_urbano_rural",
    "MAXEDUCP_A": "escolaridade_maxima_familia",
    "PHSTAT_A": "autoavaliacao_saude_geral",
    "BMICAT_A": "categoria_imc",
    "WEIGHTLBTC_A": "peso_libras",
    "HYPEV_A": "historico_hipertensao",
    "DIBPILL_A": "med_diabetes_oral",
    "ASTILL_A": "asma_ativa_atualmente",
    "CERVIAGETC_A": "idade_cancer_colo_utero",
    "LUNGAGETC_A": "idade_cancer_pulmao",
    "LIVERAGETC_A": "idade_cancer_figado",
    "SKNDKAGETC_A": "idade_cancer_pele_indeterminado",
    "THYROAGETC_A": "idade_cancer_tireoide",
    "STOMAAGETC_A": "idade_cancer_estomago",
    "SHTFLU12M_A": "vacina_gripe_ultimos_12m",
    "WELLVIS_A": "tempo_ultimo_checkup_geral",
    "WELLNESS_A": "ultima_visita_foi_checkup",
    "EMERG12MTC_A": "visitas_emergencia_hospitalar_12m",
    "URGCC12MTC_A": "visitas_pronto_atendimento_12m",
    "INJSAWDOC_A": "atendimento_medico_por_lesao",
    "SMKNOW_A": "fuma_cigarro_atualmente",
    "CIGNOW_A": "qtd_cigarros_dia",
    "SMK30D_A": "dias_fumados_mes",
    "ECIGNOW_A": "usa_cigarro_eletronico_atualmente",
    # Controles amostrais
    "HHX": "id_domicilio",
    "WTFA_A": "peso_amostral",
}

colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

# 2. Leitura otimizada direto do CSV
df_bruto = pd.read_csv("adult24.csv", usecols=colunas_para_ler, low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")

# 3. Construção do Desfecho Clínico (target_medicacao)
# 1 = Sim, 2 = Não, outros = Indeterminado
cond_sim = (df_bruto["DEPMED_A"] == 1) | (df_bruto["ANXMED_A"] == 1)
cond_nao = (df_bruto["DEPMED_A"] == 2) & (df_bruto["ANXMED_A"] == 2)
df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))

# 4. Aplicação dos Nomes Padronizados em Português
df_limpo = df_bruto.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV limpo
nome_arquivo_saida = "adult24_limpo_selecionadas.csv"
df_limpo.to_csv(nome_arquivo_saida, index=False)

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_limpo.shape[0]:,} participantes e {df_limpo.shape[1]} atributos.")
print("\nDistribuição da variável dependente (target_medicacao):")
print(df_limpo["target_medicacao"].value_counts(dropna=False, normalize=True).round(4) * 100)

Colunas a serem carregadas: 87
Base carregada: 32,629 linhas e 87 colunas.

Base limpa exportada como 'adult24_limpo_selecionadas.csv' com sucesso!
Dimensões finais: 32,629 participantes e 88 atributos.

Distribuição da variável dependente (target_medicacao):
target_medicacao
0.0    80.60
1.0    17.71
NaN     1.69
Name: proportion, dtype: float64


In [16]:
df_bruto

,RATCAT_A,K6SPD_A,MARSTAT_A,SPOUSESEX_A,BMICAT_A,WEIGHTLBTC_A,URGCC12MTC_A,EMERG12MTC_A,MAXEDUCP_A,PARSTAT_A,...,ARTHEV_A,DIBPILL_A,ASTILL_A,HYPEV_A,LSATIS4_A,PHSTAT_A,WTFA_A,HHX,POVRATTC_A,target_medicacao
0,9,2,2,NaN,3,165,0,0,5.0,1,...,2,NaN,NaN,2,1,1,5780.565,H067658,2.82,0.0
1,8,2,4,NaN,3,180,2,2,5.0,3,...,2,NaN,NaN,1,2,2,3994.244,H076577,2.01,0.0
2,7,2,1,2.0,4,215,0,0,5.0,3,...,1,NaN,NaN,1,1,2,6636.755,H019335,1.90,0.0
3,12,2,1,2.0,3,200,0,0,10.0,1,...,1,NaN,NaN,2,2,3,13767.420,H012701,4.48,0.0
4,14,2,7,NaN,3,145,4,1,9.0,3,...,2,NaN,1.0,2,2,3,18880.030,H049678,6.37,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32624,10,2,1,2.0,3,210,0,3,5.0,3,...,1,NaN,1.0,1,2,3,1348.197,H068650,3.18,0.0
32625,14,2,1,1.0,2,125,0,0,8.0,3,...,1,NaN,NaN,1,2,1,1030.453,H062337,9.80,0.0
32626,14,2,7,NaN,2,170,0,0,7.0,3,...,2,NaN,NaN,2,1,1,1648.080,H005264,6.86,0.0
32627,8,2,1,1.0,2,142,0,0,7.0,3,...,1,NaN,NaN,2,2,2,1349.389,H034334,2.44,0.0


In [17]:
df_limpo

,categoria_razao_pobreza,sofrimento_psicologico_k6,estado_civil,sexo_conjuge,categoria_imc,peso_libras,visitas_pronto_atendimento_12m,visitas_emergencia_hospitalar_12m,escolaridade_maxima_familia,status_parental,...,historico_artrite,med_diabetes_oral,asma_ativa_atualmente,historico_hipertensao,satisfacao_com_a_vida,autoavaliacao_saude_geral,peso_amostral,id_domicilio,razao_renda_pobreza,target_medicacao
0,9,2,2,NaN,3,165,0,0,5.0,1,...,2,NaN,NaN,2,1,1,5780.565,H067658,2.82,0.0
1,8,2,4,NaN,3,180,2,2,5.0,3,...,2,NaN,NaN,1,2,2,3994.244,H076577,2.01,0.0
2,7,2,1,2.0,4,215,0,0,5.0,3,...,1,NaN,NaN,1,1,2,6636.755,H019335,1.90,0.0
3,12,2,1,2.0,3,200,0,0,10.0,1,...,1,NaN,NaN,2,2,3,13767.420,H012701,4.48,0.0
4,14,2,7,NaN,3,145,4,1,9.0,3,...,2,NaN,1.0,2,2,3,18880.030,H049678,6.37,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32624,10,2,1,2.0,3,210,0,3,5.0,3,...,1,NaN,1.0,1,2,3,1348.197,H068650,3.18,0.0
32625,14,2,1,1.0,2,125,0,0,8.0,3,...,1,NaN,NaN,1,2,1,1030.453,H062337,9.80,0.0
32626,14,2,7,NaN,2,170,0,0,7.0,3,...,2,NaN,NaN,2,1,1,1648.080,H005264,6.86,0.0
32627,8,2,1,1.0,2,142,0,0,7.0,3,...,1,NaN,NaN,2,2,2,1349.389,H034334,2.44,0.0


In [18]:
df_limpo["target_medicacao"]

0        0.0
1        0.0
2        0.0
3        0.0
4        1.0
        ... 
32624    0.0
32625    0.0
32626    0.0
32627    0.0
32628    0.0
Name: target_medicacao, Length: 32629, dtype: float64